# 구조적 데이터 분할 및 ML 분류 실험

이 노트북은 피드백 데이터의 **ID 구조**(학생ID_검사지유형q질문번호_시퀀스)를 활용하여 6가지 구조적 분할 전략을 생성하고, 각 분할에 대해 24개 ML 모델의 성능을 평가합니다. 이 실험의 목적은 **모델이 새로운 맥락(새 검사지, 새 질문, 새 학생)에 얼마나 잘 일반화하는지** 검증하는 것입니다.

## 6가지 분할 전략

| 전략 | 설명 | 학생 분리 | 검증 목적 |
|------|------|:---------:|----------|
| S1 | 검사지 유형: A→train, B→test | X | 새 검사지 일반화 |
| S1+S4 | 검사지 유형 + 학생 분리 | O | 새 검사지 + 새 학생 일반화 |
| S2 | 질문 번호 기반 분할 | X | 새 질문 유형 일반화 |
| S2+S4 | 질문 번호 + 학생 분리 | O | 새 질문 + 새 학생 일반화 |
| S3 | 검사지×질문 교차 분할 | X | 복합 일반화 |
| S3+S4 | 교차 + 학생 분리 | O | 가장 엄격한 일반화 |

- **학생 분리(S4)**: 같은 학생의 데이터가 훈련과 테스트에 동시에 포함되지 않도록 분리하여, 모델이 특정 학생의 응답 패턴을 학습하는 것을 방지합니다.

## ML 모델 (24개)
- **개별 9개**: LR, SVM-L, SVM-RBF, CNB, DT, RF, GB, XGB, LGBM
- **앙상블 15개**: 5 조합 × 3 전략(Hard/Soft Voting, Stacking)

## 라이브러리 로드

데이터 처리, 형태소 분석(Kiwi), TF-IDF 벡터화, 9종의 ML 분류기 및 앙상블 기법을 위한 라이브러리를 불러옵니다.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
import os
from datetime import datetime
from copy import deepcopy
import json
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# 형태소 분석기 (한국어 텍스트 토큰화)
from kiwipiepy import Kiwi
# 텍스트 벡터화: 단어를 수치 벡터로 변환
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
# 개별 분류 알고리즘
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.tree import DecisionTreeClassifier
# 앙상블 분류 알고리즘
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    VotingClassifier,       # 투표 기반 앙상블
    StackingClassifier       # 스태킹 기반 앙상블
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
# 교차 검증 및 평가 지표
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score

print(f"실험 시작: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("라이브러리 로드 완료!")

## 1. 설정

실험의 주요 파라미터를 정의합니다.
- **분할 전략별 상세 설정**: 질문 기반(S2), 교차 분할(S3)의 구체적인 훈련/테스트 배정
- **ML 모델**: 9개 개별 모델 + 5가지 앙상블 조합 × 3가지 앙상블 방법 = 24개
- **교차 검증**: 10-Fold Stratified CV로 과적합(overfitting) 여부를 확인

In [ ]:
# === 파일 경로 ===
DATA_PATH = Path("../data/feedback_data.xlsx")
OUTPUT_DIR = Path("./structural_splits")       # 분할 파일 저장 디렉토리
RESULTS_DIR = OUTPUT_DIR / 'results'           # 실험 결과 저장 디렉토리
OUTPUT_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

# === 공통 설정 ===
RANDOM_SEED = 42       # 재현성을 위한 단일 시드
N_FOLDS = 10           # 10-Fold 교차 검증

# === 전략 2(S2): 질문 기반 분할 설정 ===
# q1~q3로 훈련, q4로 테스트 → 새로운 유형의 질문에 대한 일반화 검증
S2_TRAIN_QUESTIONS = ['q1', 'q2', 'q3']
S2_TEST_QUESTIONS = ['q4']

# === 전략 3(S3): 교차 분할 설정 ===
# A형 전체 + B형 q1,q2로 훈련, B형 q3,q4로 테스트 → 검사지와 질문의 복합 일반화
S3_TRAIN = {'a': ['q1', 'q2', 'q3', 'q4'], 'b': ['q1', 'q2']}
S3_TEST = {'b': ['q3', 'q4']}

# === ML 모델 설정 ===
INDIVIDUAL_MODELS = ['LR', 'SVM-L', 'SVM-RBF', 'CNB', 'DT', 'RF', 'GB', 'XGB', 'LGBM']

# 앙상블 조합: 선행 연구와 최적화 실험 결과에 기반한 5가지 모델 조합
ENSEMBLE_COMBOS = {
    'C1_Optimal5': ['SVM-L', 'CNB', 'DT', 'GB', 'XGB'],      # 최적 5종 조합
    'C2_Alt5':     ['LR', 'SVM-L', 'DT', 'GB', 'XGB'],        # 대안 5종 조합
    'C3_Optimal3': ['LR', 'SVM-RBF', 'XGB'],                   # 최적 3종 조합
    'C4_Prev4':    ['LR', 'SVM-L', 'CNB', 'RF'],               # 선행연구 4종
    'C5_Prev5':    ['LR', 'SVM-RBF', 'CNB', 'RF', 'LGBM'],    # 선행연구 5종
}
# 앙상블 전략: Hard Voting(다수결), Soft Voting(확률 평균), Stacking(메타 학습기)
ENSEMBLE_METHODS = ['Hard Voting', 'Soft Voting', 'Stacking']

n_models = len(INDIVIDUAL_MODELS) + len(ENSEMBLE_COMBOS) * len(ENSEMBLE_METHODS)
print(f"설정 완료")
print(f"  개별 모델: {len(INDIVIDUAL_MODELS)}개, 앙상블: {len(ENSEMBLE_COMBOS)}×{len(ENSEMBLE_METHODS)}={len(ENSEMBLE_COMBOS)*len(ENSEMBLE_METHODS)}개")
print(f"  총 모델: {n_models}개 × 6 전략 = {n_models*6}회 실험")

## 2. 데이터 로드 및 전처리

### 2.1 데이터 로드 및 ID 파싱

피드백 데이터의 ID 구조(`학생ID_검사지유형q질문번호_시퀀스`)를 파싱하여 **학생**, **검사지 유형(a/b)**, **질문 번호(q1~q4)** 정보를 추출합니다. 이 정보가 구조적 분할의 기준이 됩니다.

In [ ]:
# 데이터 로드
df = pd.read_excel(DATA_PATH)
print(f"전체 데이터: {len(df)}개, 컬럼: {df.columns.tolist()}\n")

# ID 파싱: 정규표현식으로 ID에서 학생/검사지/질문 정보 추출
df['student'] = df['id'].str.extract(r'^(\d+)_')      # 학생 코드 (가명)
df['form'] = df['id'].str.extract(r'_([ab])q')         # 검사지 유형 (a 또는 b)
df['question'] = df['id'].str.extract(r'_[ab](q\d)_')  # 질문 번호 (q1~q4)

print("=== ID 구조 파싱 ===")
print(f"학생: {df['student'].nunique()}명, 검사지: {sorted(df['form'].unique())}, 질문: {sorted(df['question'].unique())}")
print(f"\n=== 검사지 × 질문 교차표 ===")
print(pd.crosstab(df['form'], df['question'], margins=True))

# 학생 참여 현황: A형만, B형만, 양쪽 모두 참여한 학생 수
students_a = set(df[df['form']=='a']['student'].unique())
students_b = set(df[df['form']=='b']['student'].unique())
print(f"\n학생 - A형만: {len(students_a-students_b)}명, B형만: {len(students_b-students_a)}명, 양쪽: {len(students_a&students_b)}명")

print(f"\n=== 레이블 분포 ===")
print(df['label'].value_counts().sort_index())

**결과 해석:** 55명의 학생이 A형(1,060건)과 B형(1,020건) 검사지에 각 4개 질문씩 응답했습니다. 49명(89%)이 양쪽 검사지를 모두 응답하였으므로, 검사지 기반 분할(S1) 시 학생 겹침이 발생합니다. 이것이 S4(학생 분리) 전략이 필요한 이유입니다.

### 2.2 형태소 분석 (토큰화)

한국어 텍스트를 ML 모델에 입력하기 위해 **형태소 분석(morphological analysis)**을 수행합니다.
- **Kiwi**: 한국어 형태소 분석기로, 텍스트를 품사 단위로 분해
- **원형 복원(Lemmatization)**: 활용형을 기본형으로 변환 (예: "적절한" → "적절하다")
- **품사 필터링**: 명사, 동사, 형용사, 부사 등 의미 있는 품사만 추출

In [ ]:
# 형태소 분석 (Kiwi + Lemmatization)
class KiwiLemmaTokenizer:
    # 추출 대상 품사: 일반명사(NNG), 고유명사(NNP), 의존명사(NNB), 수사(NR),
    # 동사(VV), 형용사(VA), 보조용언(VX), 일반부사(MAG), 접속부사(MAJ)
    INCLUDE_POS = {'NNG','NNP','NNB','NR','VV','VA','VX','MAG','MAJ'}

    def __init__(self):
        self.kiwi = Kiwi()

    def tokenize_with_lemma(self, text):
        if pd.isna(text) or str(text).strip() == '':
            return ''
        tokens = self.kiwi.tokenize(str(text))
        # 원형(lemma)이 있으면 원형을, 없으면 표면형(form)을 사용
        return ' '.join(t.lemma or t.form for t in tokens if t.tag in self.INCLUDE_POS)

print("토큰화 중...")
tokenizer = KiwiLemmaTokenizer()
df['tokenized'] = df['feedback_text'].apply(tokenizer.tokenize_with_lemma)
print(f"토큰화 완료! (예시: '{df['tokenized'].iloc[0]}')") 

## 3. 분할 함수 정의

### 학생 분리 함수와 분할 요약 함수

- `apply_student_isolation()`: 훈련/테스트에 겹치는 학생을 원래 분할 비율에 맞게 한쪽에만 배정합니다. 이 과정에서 데이터가 일부 제거되므로 전체 활용률이 낮아집니다.
- `show_split_summary()`: 분할 결과의 크기, 학생 겹침, 레이블 분포를 한눈에 확인합니다.

In [ ]:
def apply_student_isolation(train_df, test_df, random_state=42):
    """학생 분리: 겹치는 학생을 구조적 비율에 맞게 한쪽에만 배정.
    
    예를 들어, 학생 A가 훈련과 테스트 모두에 데이터가 있으면,
    훈련/테스트 비율에 따라 학생 A를 한쪽에만 배정하고 나머지에서 제거합니다.
    """
    train_stu = set(train_df['student'].unique())
    test_stu = set(test_df['student'].unique())
    overlap = train_stu & test_stu  # 양쪽에 모두 존재하는 학생

    if not overlap:
        print("  겹치는 학생 없음.")
        return train_df.copy(), test_df.copy()

    # 원래 분할 비율에 맞게 겹치는 학생을 배정
    train_ratio = len(train_df) / (len(train_df) + len(test_df))
    rng = np.random.RandomState(random_state)
    overlap_list = sorted(list(overlap))
    rng.shuffle(overlap_list)  # 무작위로 섞어 편향 방지

    n_to_train = max(1, int(len(overlap_list) * train_ratio))
    assign_train = set(overlap_list[:n_to_train])   # 훈련에 배정할 학생
    assign_test = set(overlap_list[n_to_train:])     # 테스트에 배정할 학생

    # 배정되지 않은 학생의 데이터를 제거
    train_clean = train_df[~train_df['student'].isin(assign_test)].copy()
    test_clean = test_df[~test_df['student'].isin(assign_train)].copy()

    print(f"  겹치는 학생: {len(overlap)}명 → train {len(assign_train)}명, test {len(assign_test)}명")
    print(f"  제거: train -{len(train_df)-len(train_clean)}개, test -{len(test_df)-len(test_clean)}개")
    return train_clean, test_clean


def show_split_summary(name, train_df, test_df):
    """분할 결과 요약: 크기, 학생 겹침, 레이블 분포를 출력."""
    total = len(train_df) + len(test_df)
    train_stu = set(train_df['student'].unique())
    test_stu = set(test_df['student'].unique())
    stu_overlap = train_stu & test_stu  # 학생 겹침 확인

    print(f"\n{'='*60}")
    print(f"[{name}]")
    print(f"Train: {len(train_df)}개 ({len(train_df)/total*100:.1f}%), Test: {len(test_df)}개 ({len(test_df)/total*100:.1f}%)")
    print(f"합계: {total}개 (원본 대비 {total/len(df)*100:.1f}%)")
    print(f"학생 - Train: {len(train_stu)}명, Test: {len(test_stu)}명, 겹침: {len(stu_overlap)}명")

    # 레이블 분포 비교 (원본 vs 훈련 vs 테스트)
    comp = pd.DataFrame({
        '원본': (df['label'].value_counts(normalize=True).sort_index()*100).round(1),
        'Train': (train_df['label'].value_counts(normalize=True).sort_index()*100).round(1),
        'Test': (test_df['label'].value_counts(normalize=True).sort_index()*100).round(1)
    })
    print(comp)

    return {
        'train_n': len(train_df), 'test_n': len(test_df), 'total_n': total,
        'usage': f"{total/len(df)*100:.1f}%",
        'stu_overlap': len(stu_overlap), 'train_stu': len(train_stu), 'test_stu': len(test_stu)
    }

print("함수 정의 완료")

## 4. 전략 실행

### 전략 1 (S1): 검사지 유형(A/B) 기반 분할

A형 검사지 → 훈련, B형 검사지 → 테스트로 분할합니다. 이 전략은 모델이 **새로운 검사지**에 대해 얼마나 일반화하는지 검증합니다.

In [6]:
results = {}
splits = {}

# S1: 검사지 기반
train_s1 = df[df['form'] == 'a'].copy()
test_s1 = df[df['form'] == 'b'].copy()
results['S1'] = show_split_summary('S1: A→train, B→test', train_s1, test_s1)
splits['S1_form'] = (train_s1, test_s1)


[S1: A→train, B→test]
Train: 1060개 (51.0%), Test: 1020개 (49.0%)
합계: 2080개 (원본 대비 100.0%)
학생 - Train: 53명, Test: 51명, 겹침: 49명
         원본  Train  Test
label                   
0       2.9    3.3   2.5
1      24.7   13.7  36.1
2      13.7   13.9  13.5
3      35.1   37.5  32.6
4       8.7   11.5   5.8
5      13.3   17.6   8.8
6       1.6    2.5   0.7


**결과 해석:** S1 분할은 전체 데이터(2,080건)를 100% 활용하지만, 49명의 학생이 훈련과 테스트에 동시에 존재합니다. 또한 A형과 B형의 레이블 분포가 상당히 다릅니다(예: 레이블 1이 A형 13.7% vs B형 36.1%). 이는 두 검사지의 질문 난이도가 다르기 때문입니다.

### S1+S4: 검사지 분할 + 학생 분리

S1에 학생 분리(S4)를 적용하여, 같은 학생이 훈련과 테스트에 동시에 존재하지 않도록 합니다.

In [7]:
# S1+S4: 검사지 + 학생 분리
print("학생 분리 적용 중...")
train_s1s4, test_s1s4 = apply_student_isolation(train_s1.copy(), test_s1.copy(), RANDOM_SEED)
results['S1+S4'] = show_split_summary('S1+S4: 검사지 + 학생분리', train_s1s4, test_s1s4)
splits['S1S4_form_student'] = (train_s1s4, test_s1s4)

학생 분리 적용 중...
  겹치는 학생: 49명 → train 24명, test 25명
  제거: train -500개, test -480개

[S1+S4: 검사지 + 학생분리]
Train: 560개 (50.9%), Test: 540개 (49.1%)
합계: 1100개 (원본 대비 52.9%)
학생 - Train: 28명, Test: 27명, 겹침: 0명
         원본  Train  Test
label                   
0       2.9    3.8   3.9
1      24.7   10.2  40.6
2      13.7   17.7  11.1
3      35.1   38.9  27.8
4       8.7    7.0   6.3
5      13.3   19.8  10.4
6       1.6    2.7   NaN


**결과 해석:** 학생 분리로 인해 데이터 활용률이 100%에서 52.9%(1,100건)으로 크게 감소했습니다. 49명의 겹치는 학생을 24명(→훈련), 25명(→테스트)으로 분리하면서, 해당 학생들의 반대편 데이터가 모두 제거되기 때문입니다. 학생 겹침이 0명으로 완전 분리되었습니다.

### 전략 2 (S2): 질문 번호 기반 분할

q1, q2, q3 → 훈련, q4 → 테스트로 분할합니다. 이 전략은 모델이 **학습하지 않은 새로운 질문 유형**에 대해 얼마나 일반화하는지 검증합니다.

In [8]:
# S2: 질문 기반
print(f"Train: {S2_TRAIN_QUESTIONS}, Test: {S2_TEST_QUESTIONS}")
train_s2 = df[df['question'].isin(S2_TRAIN_QUESTIONS)].copy()
test_s2 = df[df['question'].isin(S2_TEST_QUESTIONS)].copy()
results['S2'] = show_split_summary('S2: q1,q2,q3→train, q4→test', train_s2, test_s2)
splits['S2_question'] = (train_s2, test_s2)

Train: ['q1', 'q2', 'q3'], Test: ['q4']

[S2: q1,q2,q3→train, q4→test]
Train: 1560개 (75.0%), Test: 520개 (25.0%)
합계: 2080개 (원본 대비 100.0%)
학생 - Train: 55명, Test: 55명, 겹침: 55명
         원본  Train  Test
label                   
0       2.9    2.8   3.3
1      24.7   24.2  26.2
2      13.7   14.5  11.3
3      35.1   33.4  40.4
4       8.7    9.2   7.3
5      13.3   14.3  10.4
6       1.6    1.7   1.2


**결과 해석:** S2는 75:25 비율로 분할되며 전체 데이터를 100% 활용합니다. 그러나 55명 전원이 양쪽에 존재하여 학생 겹침이 최대입니다. 레이블 분포의 차이가 S1보다 적어, 질문 간 난이도 차이가 검사지 간 차이보다 작음을 시사합니다.

### S2+S4: 질문 분할 + 학생 분리

In [9]:
# S2+S4: 질문 + 학생 분리
print("학생 분리 적용 중...")
train_s2s4, test_s2s4 = apply_student_isolation(train_s2.copy(), test_s2.copy(), RANDOM_SEED)
results['S2+S4'] = show_split_summary('S2+S4: 질문 + 학생분리', train_s2s4, test_s2s4)
splits['S2S4_question_student'] = (train_s2s4, test_s2s4)

학생 분리 적용 중...
  겹치는 학생: 55명 → train 41명, test 14명
  제거: train -390개, test -390개

[S2+S4: 질문 + 학생분리]
Train: 1170개 (90.0%), Test: 130개 (10.0%)
합계: 1300개 (원본 대비 62.5%)
학생 - Train: 41명, Test: 14명, 겹침: 0명
         원본  Train  Test
label                   
0       2.9    3.7   0.8
1      24.7   26.1  23.1
2      13.7   13.6  16.9
3      35.1   33.0  45.4
4       8.7    8.5   5.4
5      13.3   13.3   8.5
6       1.6    1.9   NaN


**결과 해석:** 학생 분리 후 활용률이 62.5%(1,300건)로 감소했으며, 테스트 세트가 130건으로 작아졌습니다. 훈련 90% : 테스트 10%의 불균형이 발생했습니다. 레이블 6이 테스트에 NaN(0건)으로 나타나 매우 희소한 레이블의 평가가 제한적입니다.

### 전략 3 (S3): 검사지 x 질문 교차 분할

A형 전체 + B형 q1,q2 → 훈련, B형 q3,q4 → 테스트로 분할합니다. 검사지와 질문을 동시에 교차시키는 가장 복합적인 일반화 전략입니다.

In [ ]:
# S3: 교차 분할 - 불리언 마스크를 사용하여 조건에 맞는 데이터 선택
print("Train:", S3_TRAIN)
print("Test: ", S3_TEST)

# 훈련 마스크: A형 q1~q4 전체 + B형 q1,q2
train_mask = pd.Series(False, index=df.index)
for form, qs in S3_TRAIN.items():
    train_mask |= (df['form'] == form) & (df['question'].isin(qs))

# 테스트 마스크: B형 q3,q4만
test_mask = pd.Series(False, index=df.index)
for form, qs in S3_TEST.items():
    test_mask |= (df['form'] == form) & (df['question'].isin(qs))

train_s3 = df[train_mask].copy()
test_s3 = df[test_mask].copy()
results['S3'] = show_split_summary('S3: A전체+Bq1q2→train, Bq3q4→test', train_s3, test_s3)
splits['S3_cross'] = (train_s3, test_s3)

**결과 해석:** S3은 75.5:24.5로 분할되며, 테스트 세트는 B형의 q3, q4만 포함합니다. 51명의 학생이 겹치므로 학생 분리가 필요합니다.

### S3+S4: 교차 분할 + 학생 분리

가장 엄격한 일반화 조건입니다. 검사지 유형, 질문 번호, 학생 모두가 훈련과 테스트에서 분리됩니다.

In [11]:
# S3+S4: 교차 + 학생 분리
print("학생 분리 적용 중...")
train_s3s4, test_s3s4 = apply_student_isolation(train_s3.copy(), test_s3.copy(), RANDOM_SEED)
results['S3+S4'] = show_split_summary('S3+S4: 교차 + 학생분리', train_s3s4, test_s3s4)
splits['S3S4_cross_student'] = (train_s3s4, test_s3s4)

학생 분리 적용 중...
  겹치는 학생: 51명 → train 38명, test 13명
  제거: train -390개, test -380개

[S3+S4: 교차 + 학생분리]
Train: 1180개 (90.1%), Test: 130개 (9.9%)
합계: 1310개 (원본 대비 63.0%)
학생 - Train: 42명, Test: 13명, 겹침: 0명
         원본  Train  Test
label                   
0       2.9    2.3   0.8
1      24.7   22.2  31.5
2      13.7   12.7  20.8
3      35.1   35.7  42.3
4       8.7    9.1   3.1
5      13.3   16.1   1.5
6       1.6    1.9   NaN


**결과 해석:** S3+S4는 가장 엄격한 분할로, 활용률 63.0%(1,310건), 테스트 130건입니다. 학생 겹침 0명으로 완전 분리되었습니다. 레이블 5(대안 제시)가 테스트에서 1.5%로 매우 적고, 레이블 6은 테스트에 없습니다(NaN).

## 5. 분할 비교 요약

6가지 전략의 핵심 특성을 한눈에 비교합니다. 주요 관심사항:
- **활용률**: 학생 분리(+S4) 시 데이터 손실 정도
- **학생 겹침**: 0명이면 완전한 학생 독립 검증이 가능
- **훈련/테스트 비율**: 너무 불균형하면 평가 신뢰성이 낮아짐

In [12]:
summary = pd.DataFrame(results).T
summary.index.name = '전략'
col_map = {'train_n':'Train','test_n':'Test','total_n':'합계','usage':'활용률','train_stu':'Train학생','test_stu':'Test학생','stu_overlap':'겹침'}
print("=" * 80)
print("전체 전략 비교")
print("=" * 80)
print(summary.rename(columns=col_map).to_string())
print(f"\n원본: {len(df)}개, {df['student'].nunique()}명")

전체 전략 비교
      Train  Test    합계     활용률  겹침 Train학생 Test학생
전략                                                
S1     1060  1020  2080  100.0%  49      53     51
S1+S4   560   540  1100   52.9%   0      28     27
S2     1560   520  2080  100.0%  55      55     55
S2+S4  1170   130  1300   62.5%   0      41     14
S3     1570   510  2080  100.0%  51      55     51
S3+S4  1180   130  1310   63.0%   0      42     13

원본: 2080개, 55명


**결과 해석:** 학생 분리가 없는 전략(S1, S2, S3)은 데이터를 100% 활용하지만 학생 겹침이 있습니다. 학생 분리를 적용하면(+S4) 겹침은 0이 되지만 활용률이 52.9~63.0%로 감소합니다. 이는 **일반화 엄격성과 데이터 활용의 트레이드오프(trade-off)**를 보여줍니다.

## 6. 분할 파일 저장

각 전략별 훈련/테스트 데이터를 Excel 파일로 저장합니다. 총 12개 파일(6전략 × 2세트)이 생성됩니다.

In [ ]:
# 저장할 컬럼: 원본 3개 컬럼만 저장 (파싱된 컬럼은 필요 시 재생성 가능)
save_cols = ['id', 'feedback_text', 'label']
print(f"저장 경로: {OUTPUT_DIR.resolve()}\n")

for name, (train, test) in splits.items():
    train[save_cols].to_excel(OUTPUT_DIR / f"train_{name}.xlsx", index=False)
    test[save_cols].to_excel(OUTPUT_DIR / f"test_{name}.xlsx", index=False)
    print(f"[{name}] train={len(train)}개, test={len(test)}개")

print(f"\n총 {len(splits)*2}개 파일 저장 완료!")

In [ ]:
# 저장 파일 검증: 크기와 데이터 누출 여부 확인
print("=== 파일 검증 ===")
for name, (tr_orig, te_orig) in splits.items():
    tr = pd.read_excel(OUTPUT_DIR / f"train_{name}.xlsx")
    te = pd.read_excel(OUTPUT_DIR / f"test_{name}.xlsx")
    # 크기 일치 + ID 중복 없음 = 정상
    ok = len(tr)==len(tr_orig) and len(te)==len(te_orig) and len(set(tr['id'])&set(te['id']))==0
    print(f"[{'OK' if ok else 'FAIL'}] {name}: train={len(tr)}, test={len(te)}")
print("검증 완료!")

---
# Part 2: ML 분류 실험

여기서부터는 위에서 생성한 6가지 분할에 대해 24개 ML 모델을 학습하고 평가합니다.

## 7. 분류기 및 실험 함수 정의

### 7.1 텍스트 벡터화 (TF-IDF)

**TF-IDF**(Term Frequency-Inverse Document Frequency): 단어의 빈도와 역문서 빈도를 곱하여 각 단어의 중요도를 수치로 변환하는 방법입니다. 
- 단어 단위(word-level) + 1~2 연속 단어(n-gram) 조합
- `sublinear_tf=True`: TF에 로그 스케일링을 적용하여 매우 빈번한 단어의 영향을 완화

### 7.2 분류기 정의

9개 개별 분류기와 15개 앙상블(5조합 × 3방법)을 정의합니다.
- **앙상블(ensemble)**: 여러 분류기의 예측을 결합하여 성능을 향상시키는 기법
  - **Hard Voting**: 다수결 투표
  - **Soft Voting**: 확률 평균
  - **Stacking**: 기본 분류기의 출력을 메타 학습기가 재학습

In [ ]:
def create_vectorizer():
    """V1_Word: Word TF-IDF (lemma, ngram 1-2)"""
    return TfidfVectorizer(
        tokenizer=lambda x: x.split() if x else [],  # 이미 토큰화된 텍스트를 공백으로 분리
        preprocessor=lambda x: x,                     # 전처리 비활성화 (이미 완료)
        ngram_range=(1, 2),   # 단일 단어 + 2단어 연속 조합
        min_df=2,             # 최소 2개 문서에 출현한 단어만 사용
        max_df=0.95,          # 95% 이상 문서에 출현하는 단어 제외 (불용어 역할)
        sublinear_tf=True     # TF에 로그 스케일링 적용
    )


def get_single_classifier(name, random_state=42):
    """9개 개별 분류기 정의. class_weight='balanced'로 레이블 불균형에 대응."""
    clfs = {
        'LR': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=random_state, n_jobs=-1),
        'SVM-L': LinearSVC(class_weight='balanced', max_iter=2000, random_state=random_state),
        'SVM-RBF': SVC(kernel='rbf', class_weight='balanced', random_state=random_state, probability=True),
        'CNB': ComplementNB(alpha=1.0),  # 보완 나이브 베이즈: 불균형 데이터에 적합
        'DT': DecisionTreeClassifier(class_weight='balanced', random_state=random_state),
        'RF': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=random_state, n_jobs=-1),
        'GB': GradientBoostingClassifier(n_estimators=100, random_state=random_state),
        'XGB': XGBClassifier(n_estimators=100, eval_metric='mlogloss', random_state=random_state, n_jobs=-1, verbosity=0),
        'LGBM': LGBMClassifier(n_estimators=100, class_weight='balanced', random_state=random_state, n_jobs=-1, verbose=-1),
    }
    return clfs[name]


def get_soft_classifier(name, random_state=42):
    """Soft Voting/Stacking용: LinearSVC는 확률 출력이 불가하므로 SVC로 대체."""
    if name == 'SVM-L':
        return SVC(kernel='linear', class_weight='balanced', random_state=random_state, probability=True)
    return get_single_classifier(name, random_state)


def build_ensemble(combo_models, method, random_state=42):
    """앙상블 분류기 구성: 조합 + 방법에 따라 적절한 앙상블 생성."""
    if method == 'Hard Voting':
        est = [(n.lower().replace('-','_'), get_single_classifier(n, random_state)) for n in combo_models]
        return VotingClassifier(estimators=est, voting='hard', n_jobs=-1)
    elif method == 'Soft Voting':
        est = [(n.lower().replace('-','_'), get_soft_classifier(n, random_state)) for n in combo_models]
        return VotingClassifier(estimators=est, voting='soft', n_jobs=-1)
    elif method == 'Stacking':
        est = [(n.lower().replace('-','_'), get_soft_classifier(n, random_state)) for n in combo_models]
        return StackingClassifier(
            estimators=est,
            final_estimator=LogisticRegression(class_weight='balanced', max_iter=1000, random_state=random_state),
            cv=5, n_jobs=-1  # 5-Fold CV로 메타 특징 생성
        )

print("분류기 정의 완료")
print(f"  개별: {INDIVIDUAL_MODELS}")
print(f"  앙상블 조합: {list(ENSEMBLE_COMBOS.keys())}")
print(f"  앙상블 전략: {ENSEMBLE_METHODS}")

### 7.3 교차 검증 및 평가 함수

- **Stratified K-Fold CV**: 훈련 데이터를 K개 폴드로 나누어 K번 학습/검증을 반복하는 방법. 층화(stratified)는 각 폴드의 레이블 비율을 유지합니다.
- **평가 지표**:
  - **Accuracy**: 전체 정확도
  - **F1 Macro**: 각 레이블별 F1 점수의 단순 평균 (불균형 데이터에서 소수 레이블도 동등하게 반영)
  - **Cohen's Kappa**: 우연에 의한 일치를 보정한 일치도 지표 (0.6 이상이면 substantial agreement)

In [ ]:
def run_stratified_cv(clf, X_train, y_train, n_splits=10, random_state=42):
    """Stratified K-Fold CV 수행: 훈련 데이터 내에서 모델 성능을 추정."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    scores = {'acc': [], 'f1': [], 'kappa': []}

    for tr_idx, val_idx in skf.split(X_train, y_train):
        clf_c = deepcopy(clf)                           # 매 폴드마다 새로운 분류기 복사본 사용
        clf_c.fit(X_train[tr_idx], y_train[tr_idx])     # 훈련 폴드로 학습
        y_pred = clf_c.predict(X_train[val_idx])         # 검증 폴드로 예측
        scores['acc'].append(accuracy_score(y_train[val_idx], y_pred))
        scores['f1'].append(f1_score(y_train[val_idx], y_pred, average='macro'))
        scores['kappa'].append(cohen_kappa_score(y_train[val_idx], y_pred))

    # K번의 결과를 평균 및 표준편차로 요약
    return {
        'CV_Acc': np.mean(scores['acc']), 'CV_Acc_Std': np.std(scores['acc']),
        'CV_F1': np.mean(scores['f1']), 'CV_F1_Std': np.std(scores['f1']),
        'CV_Kappa': np.mean(scores['kappa']), 'CV_Kappa_Std': np.std(scores['kappa']),
    }


def evaluate_test(y_true, y_pred):
    """테스트 세트 성능 평가: 최종 모델의 일반화 성능을 측정."""
    return {
        'Test_Acc': accuracy_score(y_true, y_pred),
        'Test_F1': f1_score(y_true, y_pred, average='macro'),      # 레이블별 동등 가중
        'Test_F1_W': f1_score(y_true, y_pred, average='weighted'),  # 빈도 가중
        'Test_Kappa': cohen_kappa_score(y_true, y_pred),
    }

print("실험 함수 정의 완료")

## 8. 전체 실험 실행

6개 분할 전략 × 24개 모델 = **144회 실험**을 수행합니다. 각 실험에서:
1. 해당 분할의 훈련 데이터를 TF-IDF로 벡터화
2. 10-Fold Stratified CV로 훈련 데이터 내 성능 추정
3. 전체 훈련 데이터로 최종 모델 학습 후 테스트 세트에서 일반화 성능 평가

> **주의**: 실험 소요 시간은 약 30~60분입니다 (Stacking 앙상블이 특히 느림).

In [17]:
# 레이블 인코더
label_encoder = LabelEncoder()
label_encoder.fit(df['label'])
print(f"레이블 클래스: {label_encoder.classes_}")

# 전체 실험
total_exp = len(splits) * (len(INDIVIDUAL_MODELS) + len(ENSEMBLE_COMBOS) * len(ENSEMBLE_METHODS))
all_results = []
all_predictions = {}

print(f"\n{'='*70}")
print(f"전체 실험 시작: {total_exp}회")
print(f"{'='*70}")

pbar = tqdm(total=total_exp, desc="전체 진행률")

for strategy_name, (train_split, test_split) in splits.items():
    print(f"\n{'#'*70}")
    print(f"# 전략: {strategy_name} (Train={len(train_split)}, Test={len(test_split)})")
    print(f"{'#'*70}")

    # 벡터화 (V1_Word)
    vectorizer = create_vectorizer()
    X_train = vectorizer.fit_transform(train_split['tokenized'])
    X_test = vectorizer.transform(test_split['tokenized'])
    y_train = label_encoder.transform(train_split['label'])
    y_test = label_encoder.transform(test_split['label'])
    print(f"  X_train: {X_train.shape}, X_test: {X_test.shape}")

    # 예측 저장용 DataFrame
    pred_df = test_split[['id', 'feedback_text', 'label']].copy()
    pred_df = pred_df.rename(columns={'label': 'true_label'})

    # === 개별 모델 ===
    print(f"\n  [개별 모델 {len(INDIVIDUAL_MODELS)}개]")
    for model_name in INDIVIDUAL_MODELS:
        clf = get_single_classifier(model_name, RANDOM_SEED)
        cv = run_stratified_cv(clf, X_train, y_train, N_FOLDS, RANDOM_SEED)

        clf_final = deepcopy(clf)
        clf_final.fit(X_train, y_train)
        y_pred = clf_final.predict(X_test)
        test_m = evaluate_test(y_test, y_pred)

        pred_df[f'{model_name}_pred'] = label_encoder.inverse_transform(y_pred)

        all_results.append({
            'Strategy': strategy_name, 'Type': 'Individual', 'Model': model_name,
            **cv, **test_m
        })
        pbar.update(1)
        pbar.set_postfix(s=strategy_name, m=model_name, f1=f"{test_m['Test_F1']:.3f}")

    # === 앙상블 모델 ===
    print(f"  [앙상블 {len(ENSEMBLE_COMBOS)*len(ENSEMBLE_METHODS)}개]")
    for combo_name, combo_models in ENSEMBLE_COMBOS.items():
        for method in ENSEMBLE_METHODS:
            exp_name = f"{combo_name}_{method.replace(' ', '')}"
            try:
                ens_clf = build_ensemble(combo_models, method, RANDOM_SEED)
                cv = run_stratified_cv(ens_clf, X_train, y_train, N_FOLDS, RANDOM_SEED)

                ens_final = deepcopy(ens_clf)
                ens_final.fit(X_train, y_train)
                y_pred = ens_final.predict(X_test)
                test_m = evaluate_test(y_test, y_pred)

                pred_df[exp_name] = label_encoder.inverse_transform(y_pred)

                all_results.append({
                    'Strategy': strategy_name, 'Type': 'Ensemble',
                    'Model': exp_name, 'Combo': combo_name, 'Method': method,
                    **cv, **test_m
                })
            except Exception as e:
                print(f"    오류: {exp_name} - {e}")
            pbar.update(1)
            pbar.set_postfix(s=strategy_name, m=exp_name[:20])

    all_predictions[strategy_name] = pred_df
    print(f"  완료! 예측 컬럼: {len(pred_df.columns)-3}개")

pbar.close()
print(f"\n{'='*70}")
print(f"전체 실험 완료! ({len(all_results)}개 결과)")
print(f"완료 시간: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*70}")

레이블 클래스: [0 1 2 3 4 5 6]

전체 실험 시작: 144회


전체 진행률:   0%|          | 0/144 [00:00<?, ?it/s]


######################################################################
# 전략: S1_form (Train=1060, Test=1020)
######################################################################
  X_train: (1060, 866), X_test: (1020, 866)

  [개별 모델 9개]
  [앙상블 15개]


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  완료! 예측 컬럼: 24개

######################################################################
# 전략: S1S4_form_student (Train=560, Test=540)
######################################################################
  X_train: (560, 559), X_test: (540, 559)

  [개별 모델 9개]
  [앙상블 15개]


...

  완료! 예측 컬럼: 24개

######################################################################
# 전략: S3_cross (Train=1570, Test=510)
######################################################################
  X_train: (1570, 1039), X_test: (510, 1039)

  [개별 모델 9개]
  [앙상블 15개]


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  완료! 예측 컬럼: 24개

######################################################################
# 전략: S3S4_cross_student (Train=1180, Test=130)
######################################################################
  X_train: (1180, 838), X_test: (130, 838)

  [개별 모델 9개]
  [앙상블 15개]


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  완료! 예측 컬럼: 24개

전체 실험 완료! (144개 결과)
완료 시간: 2026-03-01 02:10:56


## 9. 결과 분석

### 9.1 전략별 최고 성능 모델

각 분할 전략에서 Test Kappa가 가장 높은 모델을 확인합니다. Kappa를 기준으로 하는 이유는 레이블 불균형 상황에서 우연에 의한 일치를 보정하기 때문입니다.

In [26]:
# 전체 결과 DataFrame
results_df = pd.DataFrame(all_results)
print(f"전체 결과: {results_df.shape}")

# 전략별 최고 성능 모델
print("\n" + "=" * 90)
print("전략별 최고 성능 모델 (Test Kappa 기준)")
print("=" * 90)

for strategy in splits.keys():
    subset = results_df[results_df['Strategy'] == strategy]
    best = subset.loc[subset['Test_Kappa'].idxmax()]
    print(f"\n[{strategy}] Train={results[list(results.keys())[list(splits.keys()).index(strategy)]]['train_n']}, Test={results[list(results.keys())[list(splits.keys()).index(strategy)]]['test_n']}")
    print(f"  최고: {best['Model']}")
    print(f"  CV F1={best['CV_F1']:.4f}(±{best['CV_F1_Std']:.4f}), Test F1={best['Test_F1']:.4f}, Kappa={best['Test_Kappa']:.4f}")

전체 결과: (144, 15)

전략별 최고 성능 모델 (Test Kappa 기준)

[S1_form] Train=1060, Test=1020
  최고: C5_Prev5_Stacking
  CV F1=0.8183(±0.0377), Test F1=0.5773, Kappa=0.6442

[S1S4_form_student] Train=560, Test=540
  최고: C5_Prev5_Stacking
  CV F1=0.7902(±0.0817), Test F1=0.5969, Kappa=0.5618

[S2_question] Train=1560, Test=520
  최고: C2_Alt5_SoftVoting
  CV F1=0.7079(±0.0401), Test F1=0.5617, Kappa=0.7399

[S2S4_question_student] Train=1170, Test=130
  최고: C1_Optimal5_SoftVoting
  CV F1=0.7233(±0.0513), Test F1=0.5282, Kappa=0.7092

[S3_cross] Train=1570, Test=510
  최고: C3_Optimal3_SoftVoting
  CV F1=0.7015(±0.0582), Test F1=0.5942, Kappa=0.7897

[S3S4_cross_student] Train=1180, Test=130
  최고: C1_Optimal5_SoftVoting
  CV F1=0.6856(±0.0524), Test F1=0.6458, Kappa=0.9199


**결과 해석:** 앙상블 모델(특히 Stacking, Soft Voting)이 대부분의 전략에서 최고 성능을 달성했습니다. S3+S4(교차+학생분리)에서 Kappa=0.92로 가장 높은데, 이는 테스트 세트가 130건으로 작아 변동성이 큰 결과일 수 있습니다.

### 9.2 전략 x 모델 유형별 평균 성능

In [19]:
# 전략 × 모델유형 평균 성능 비교
print("=" * 70)
print("전략 × 모델유형별 평균 Test F1 Macro")
print("=" * 70)
pivot_type = results_df.pivot_table(index='Strategy', columns='Type', values='Test_F1', aggfunc='mean')
pivot_type['전체평균'] = results_df.groupby('Strategy')['Test_F1'].mean()
print(pivot_type.round(4).to_string())

# 전략 × 모델유형 평균 Kappa
print(f"\n전략 × 모델유형별 평균 Test Kappa")
pivot_kappa = results_df.pivot_table(index='Strategy', columns='Type', values='Test_Kappa', aggfunc='mean')
pivot_kappa['전체평균'] = results_df.groupby('Strategy')['Test_Kappa'].mean()
print(pivot_kappa.round(4).to_string())

전략 × 모델유형별 평균 Test F1 Macro
Type                   Ensemble  Individual    전체평균
Strategy                                           
S1S4_form_student        0.5263      0.4624  0.5023
S1_form                  0.5532      0.5088  0.5366
S2S4_question_student    0.5189      0.4858  0.5065
S2_question              0.6075      0.5849  0.5990
S3S4_cross_student       0.5436      0.5019  0.5280
S3_cross                 0.6103      0.5812  0.5994

전략 × 모델유형별 평균 Test Kappa
Type                   Ensemble  Individual    전체평균
Strategy                                           
S1S4_form_student        0.5128      0.4409  0.4858
S1_form                  0.5813      0.5338  0.5635
S2S4_question_student    0.6324      0.6072  0.6230
S2_question              0.7160      0.6927  0.7073
S3S4_cross_student       0.7603      0.7237  0.7466
S3_cross                 0.6848      0.6745  0.6809


**결과 해석:** 
- **F1 기준**: S3(교차)과 S2(질문 기반)가 평균 F1 ~0.60으로 가장 높고, S1+S4(검사지+학생분리)가 0.50으로 가장 낮습니다.
- **Kappa 기준**: S3+S4(교차+학생분리)가 평균 0.75로 가장 높습니다. 이는 테스트 세트가 작아(130건) Accuracy가 높기 때문입니다.
- 전반적으로 앙상블이 개별 모델보다 평균 2~5%p 높은 성능을 보입니다.

### 9.3 전략별 상위 5개 모델

In [27]:
# 상위 10개 모델 (전체 전략 통합)
print("=" * 90)
print("전략별 상위 5개 모델")
print("=" * 90)

for strategy in splits.keys():
    subset = results_df[results_df['Strategy'] == strategy].sort_values('Test_Kappa', ascending=False).head(5)
    print(f"\n[{strategy}]")
    print(subset[['Model','Type','CV_F1','CV_F1_Std','Test_F1','Test_Kappa','Test_Acc']].to_string(index=False))

전략별 상위 5개 모델

[S1_form]
                 Model       Type    CV_F1  CV_F1_Std  Test_F1  Test_Kappa  Test_Acc
     C5_Prev5_Stacking   Ensemble 0.818330   0.037710 0.577310    0.644227  0.722549
  C3_Optimal3_Stacking   Ensemble 0.808786   0.039648 0.547720    0.619596  0.701961
C3_Optimal3_SoftVoting   Ensemble 0.801842   0.046605 0.545640    0.586976  0.673529
                   XGB Individual 0.751385   0.047970 0.547326    0.585810  0.669608
   C4_Prev4_SoftVoting   Ensemble 0.812300   0.051902 0.574475    0.583390  0.667647

[S1S4_form_student]
               Model     Type    CV_F1  CV_F1_Std  Test_F1  Test_Kappa  Test_Acc
   C5_Prev5_Stacking Ensemble 0.790199   0.081656 0.596887    0.561837  0.653704
C3_Optimal3_Stacking Ensemble 0.808315   0.076220 0.575387    0.542454  0.637037
   C4_Prev4_Stacking Ensemble 0.806233   0.075821 0.511190    0.540733  0.627778
 C4_Prev4_SoftVoting Ensemble 0.782670   0.108782 0.508184    0.537791  0.627778
    C2_Alt5_Stacking Ensemble 0.770315  

### 9.4 개별 모델별 전략 간 성능 변화

각 개별 모델이 6가지 분할 전략에서 보이는 성능 변동을 비교합니다. 표준편차(Std)가 작을수록 전략에 관계없이 안정적인 모델입니다.

In [21]:
# 개별 모델별 전략 간 성능 변화
print("=" * 90)
print("개별 모델별 전략 간 Test F1 비교")
print("=" * 90)

indiv = results_df[results_df['Type'] == 'Individual']
pivot_models = indiv.pivot_table(index='Model', columns='Strategy', values='Test_F1')
pivot_models['평균'] = pivot_models.mean(axis=1)
pivot_models['Std'] = pivot_models.drop(columns='평균').std(axis=1)
print(pivot_models.round(4).sort_values('평균', ascending=False).to_string())

개별 모델별 전략 간 Test F1 비교
Strategy  S1S4_form_student  S1_form  S2S4_question_student  S2_question  S3S4_cross_student  S3_cross      평균     Std
Model                                                                                                                 
RF                   0.5647   0.5334                 0.4931       0.6278              0.5127    0.5758  0.5513  0.0486
XGB                  0.5495   0.5473                 0.5053       0.4715              0.6209    0.5899  0.5474  0.0543
SVM-L                0.4560   0.5367                 0.5275       0.6519              0.4833    0.6230  0.5464  0.0769
GB                   0.4850   0.5262                 0.4829       0.5091              0.5724    0.5809  0.5261  0.0424
LR                   0.4585   0.5429                 0.4031       0.6495              0.4754    0.6097  0.5232  0.0946
LGBM                 0.4640   0.4758                 0.5079       0.5609              0.5521    0.5477  0.5181  0.0416
SVM-RBF              0.41

**결과 해석:** RF(Random Forest)가 평균 F1=0.55로 가장 높고 Std=0.049로 안정적입니다. CNB(Complement Naive Bayes)는 평균 F1=0.48로 가장 낮으며 Std=0.11로 변동이 큽니다. SVM-RBF와 LR도 전략 간 변동(Std ~0.10)이 크므로, 구조적 분할 조건에 민감한 모델입니다.

## 10. 결과 저장

실험 결과(CSV), 예측값(Excel), 실험 설정(JSON)을 파일로 저장합니다.

In [22]:
# 전체 결과 CSV 저장
results_df.to_csv(RESULTS_DIR / 'all_results.csv', index=False)
print(f"결과 저장: {RESULTS_DIR / 'all_results.csv'} ({results_df.shape})")

# 전략별 예측 파일 저장
for strategy_name, pred_df in all_predictions.items():
    filepath = RESULTS_DIR / f'predictions_{strategy_name}.xlsx'
    pred_df.to_excel(filepath, index=False)
    print(f"예측 저장: {filepath} ({pred_df.shape})")

# 실험 설정 저장
config = {
    'Date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'Random_Seed': RANDOM_SEED,
    'CV_Folds': N_FOLDS,
    'Vectorization': 'V1_Word (TF-IDF, lemma, ngram 1-2)',
    'Individual_Models': INDIVIDUAL_MODELS,
    'Ensemble_Combos': {k: v for k, v in ENSEMBLE_COMBOS.items()},
    'Ensemble_Methods': ENSEMBLE_METHODS,
    'Strategies': {k: {'train_n': v['train_n'], 'test_n': v['test_n']} for k, v in zip(results.keys(), [results[k] for k in results])},
    'Total_Experiments': len(all_results),
}
with open(RESULTS_DIR / 'experiment_config.json', 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print(f"\n설정 저장: {RESULTS_DIR / 'experiment_config.json'}")

결과 저장: structural_splits/results/all_results.csv ((144, 15))
예측 저장: structural_splits/results/predictions_S1_form.xlsx ((1020, 27))
예측 저장: structural_splits/results/predictions_S1S4_form_student.xlsx ((540, 27))
예측 저장: structural_splits/results/predictions_S2_question.xlsx ((520, 27))
예측 저장: structural_splits/results/predictions_S2S4_question_student.xlsx ((130, 27))
예측 저장: structural_splits/results/predictions_S3_cross.xlsx ((510, 27))
예측 저장: structural_splits/results/predictions_S3S4_cross_student.xlsx ((130, 27))

설정 저장: structural_splits/results/experiment_config.json


In [23]:
# 최종 요약
print("\n" + "=" * 70)
print("실험 완료 종합 요약")
print("=" * 70)

print(f"\n총 실험: {len(all_results)}회 (6 전략 × 24 모델)")
print(f"벡터화: V1_Word, Seed: {RANDOM_SEED}, CV: {N_FOLDS}-Fold")

print(f"\n[전략별 최고 F1 Macro]")
for strategy in splits.keys():
    subset = results_df[results_df['Strategy'] == strategy]
    best = subset.loc[subset['Test_F1'].idxmax()]
    print(f"  {strategy:25s} → {best['Model']:30s} F1={best['Test_F1']:.4f}  K={best['Test_Kappa']:.4f}")

print(f"\n[저장 파일]")
print(f"  분할 파일: {OUTPUT_DIR}/ (12개 xlsx)")
print(f"  결과:     {RESULTS_DIR / 'all_results.csv'}")
print(f"  예측:     {RESULTS_DIR}/predictions_*.xlsx (6개)")
print(f"  설정:     {RESULTS_DIR / 'experiment_config.json'}")
print(f"\n완료 시간: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


실험 완료 종합 요약

총 실험: 144회 (6 전략 × 24 모델)
벡터화: V1_Word, Seed: 42, CV: 10-Fold

[전략별 최고 F1 Macro]
  S1_form                   → C5_Prev5_Stacking              F1=0.5773  K=0.6442
  S1S4_form_student         → C5_Prev5_Stacking              F1=0.5969  K=0.5618
  S2_question               → C4_Prev4_Stacking              F1=0.6661  K=0.7394
  S2S4_question_student     → C5_Prev5_Stacking              F1=0.6254  K=0.6821
  S3_cross                  → C3_Optimal3_HardVoting         F1=0.6416  K=0.6955
  S3S4_cross_student        → C1_Optimal5_SoftVoting         F1=0.6458  K=0.9199

[저장 파일]
  분할 파일: structural_splits/ (12개 xlsx)
  결과:     structural_splits/results/all_results.csv
  예측:     structural_splits/results/predictions_*.xlsx (6개)
  설정:     structural_splits/results/experiment_config.json

완료 시간: 2026-03-01 02:10:56


**결과 해석:** 144회 실험이 완료되었습니다. 전략별 최고 F1 Macro는 0.577~0.666으로, 80:20 무작위 분할(F1~0.77)보다 낮습니다. 이는 구조적 분할이 의도적으로 더 어려운 일반화 조건을 만들기 때문입니다. 특히 검사지 기반(S1) 분할의 성능이 가장 낮아, A형과 B형의 피드백 특성이 상당히 다름을 시사합니다.

> **다음 단계**: 이 구조적 분할 데이터를 BERT 모델과 LLM(GPT-4o)에도 적용하여 모델 간 일반화 성능을 비교할 수 있습니다.